<a href="https://colab.research.google.com/github/Isabella-de-Lima/python_sis_26_2/blob/isabella_lima_2_bimestre/aula_11/isabella_lima_aula11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AULA 11 | ATIVIDADE | Extrator de Dados Fiscais: XML para Excel

# Você integra a equipe de Governança e Automação de TI.
# O departamento fiscal recebe arquivos XML de Notas Fiscais Eletrônicas e
# precisa consolidar os dados das compras da empresa em uma única planilha Excel
# (.xlsx) para auditoria de pagamentos.
# Requisitos:

# Pesquise sobre leitura/escrita de XML em Python e crie scripts em Python que executem as etapas a seguir:

# Referências da pesquisa:
# XML: https://docs.python.org/3/library/xml.etree.elementtree.html
# Excel: https://openpyxl.readthedocs.io/en/stable/tutorial.html

# Etapa 1: Importações e Padrão PEP 8.
# Importe os módulos nativos, de terceiros e essenciais para seu uso.
# mantendo a separação recomendada pela PEP 8.

%pip install -q openpyxl

from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path
import xml.etree.ElementTree as et

from openpyxl import Workbook, load_workbook

# Etapa 2: Coleta e Varredura de XMLs.

# Crie uma pasta com pelo menos 3 arquivos .xml de notas fiscais
# (ou utilize amostras públicas/simuladas de NF-e).
# Seu script deve percorrer a pasta lendo arquivo por arquivo.

pasta_xml = Path("notas_xml")
pasta_xml.mkdir(exist_ok=True)

notas_amostra = [
    {"numero": "1001", "data": "2026-10-10",
     "fornecedor": "Tech Solutions LTDA",
     "cnpj": "12.345.678/0001-90", "total": "4500.00"},
    {"numero": "1002", "data": "2026-10-12",
     "fornecedor": "Cloud Cloud Servicos",
     "cnpj": "98.765.432/0001-11", "total": "1280.50"},
    {"numero": "1003", "data": "2026-10-15",
     "fornecedor": "Hardware & Redes SA",
     "cnpj": "45.123.890/0001-55", "total": "8900.00"},
]

for nota in notas_amostra:
    raiz = et.Element("NFe")
    informacoes = et.SubElement(raiz, "infNFe")
    identificacao = et.SubElement(informacoes, "ide")
    et.SubElement(identificacao, "nNF").text = nota["numero"]
    et.SubElement(identificacao, "dhEmi").text = (
        f"{nota['data']}T14:30:00-03:00"
    )
    emitente = et.SubElement(informacoes, "emit")
    et.SubElement(emitente, "CNPJ").text = nota["cnpj"]
    et.SubElement(emitente, "xNome").text = nota["fornecedor"]
    total = et.SubElement(informacoes, "total")
    impostos = et.SubElement(total, "ICMSTot")
    et.SubElement(impostos, "vNF").text = nota["total"]

    arvore = et.ElementTree(raiz)
    et.indent(arvore, space="    ")
    caminho = pasta_xml / f"nfe_{nota['numero']}.xml"
    arvore.write(caminho, encoding="utf-8", xml_declaration=True)

arquivos_xml = sorted(pasta_xml.glob("*.xml"))

print("\n📂 ARQUIVOS XML ENCONTRADOS\n")

for arquivo in arquivos_xml:
    print(f"📄 {arquivo.name}")

print(f"\nQuantidade de arquivos: {len(arquivos_xml)}\n")

# Etapa 3: Extração

# Para cada nota fiscal encontrada, extraia com tratamento de erro (try/except) pelo menos as seguintes informações:
# Número da Nota
# Data de Emissão
# Nome do Fornecedor / Emitente
# CNPJ do Emitente
# Valor Total da Nota

def buscar_campo(raiz, caminho):
    """Obtém o texto de um campo obrigatório."""
    elemento = raiz.find(caminho)
    if elemento is None or not elemento.text or not elemento.text.strip():
        raise ValueError(f"Campo ausente ou vazio: {caminho}")
    return elemento.text.strip()


dados_notas = []

for arquivo in arquivos_xml:
    try:
        raiz = et.parse(arquivo).getroot()
        numero = buscar_campo(raiz, ".//{*}ide/{*}nNF")
        data_texto = buscar_campo(raiz, ".//{*}ide/{*}dhEmi")
        fornecedor = buscar_campo(raiz, ".//{*}emit/{*}xNome")
        cnpj = buscar_campo(raiz, ".//{*}emit/{*}CNPJ")
        valor_texto = buscar_campo(raiz, ".//{*}ICMSTot/{*}vNF")

        data = datetime.fromisoformat(data_texto).date()
        valor_total = Decimal(valor_texto)
        if not valor_total.is_finite() or valor_total < 0:
            raise ValueError("Valor total inválido.")

        dados_notas.append([
            numero, data.strftime("%d/%m/%Y"), fornecedor,
            cnpj, float(f"{valor_total:.2f}"),
        ])
        print(f"✅ Nota {numero} processada")
        print(f"Fornecedor: {fornecedor}")
        print(f"Valor total: R$ {valor_total:.2f}\n")

    except (et.ParseError, OSError, ValueError, InvalidOperation) as erro:
        print(f"Erro em {arquivo.name}: {erro}")

# Etapa 4: Geração da Planilha.

# Crie um arquivo csv ou um arquivo xlsx.
# Insira as linhas coletadas e formate o valor monetário com duas casas decimais (:.2f).
# Salve o arquivo .

nome_arquivo = "isabella_lima_aula11.xlsx"

if dados_notas:
    planilha = Workbook()
    aba = planilha.active
    aba.title = "Notas fiscais"
    aba.append([
        "Número da Nota", "Data de Emissão", "Fornecedor",
        "CNPJ do Emitente", "Valor Total",
    ])

    for nota in dados_notas:
        aba.append(nota)

    for linha in aba.iter_rows(min_row=2, min_col=5, max_col=5):
        linha[0].number_format = '"R$" #,##0.00'

    larguras = {"A": 20, "B": 20, "C": 30, "D": 25, "E": 20}
    for coluna, largura in larguras.items():
        aba.column_dimensions[coluna].width = largura

    planilha.save(nome_arquivo)
    print("\n📊 PLANILHA GERADA COM SUCESSO\n")
    print(f"Arquivo: {nome_arquivo}")
    print(f"Notas processadas: {len(dados_notas)}\n")
else:
    print("Nenhuma nota válida foi encontrada.")

# CONFERÊNCIA DO ARQUIVO SALVO

if dados_notas:
    arquivo_excel = load_workbook(nome_arquivo, data_only=True)
    aba_conferencia = arquivo_excel.active
    linhas = list(aba_conferencia.iter_rows(min_row=2, values_only=True))
    total_notas = sum(Decimal(str(linha[4])) for linha in linhas)

    print("\n📋 RELATÓRIO DE NOTAS FISCAIS\n")

    for linha in linhas:
        print(f"🧾 Nota fiscal: {linha[0]}")
        print(f"Data de emissão: {linha[1]}")
        print(f"Fornecedor: {linha[2]}")
        print(f"CNPJ: {linha[3]}")
        print(f"Valor total: R$ {linha[4]:.2f}\n")

    print("\n✅ CONFERÊNCIA CONCLUÍDA")
    print(f"Notas processadas: {len(linhas)}")
    print(f"Valor total das compras: R$ {total_notas:.2f}")
    print(f"Arquivo salvo: {nome_arquivo}")

    arquivo_excel.close()


📂 ARQUIVOS XML ENCONTRADOS

📄 nfe_1001.xml
📄 nfe_1002.xml
📄 nfe_1003.xml

Quantidade de arquivos: 3

✅ Nota 1001 processada
Fornecedor: Tech Solutions LTDA
Valor total: R$ 4500.00

✅ Nota 1002 processada
Fornecedor: Cloud Cloud Servicos
Valor total: R$ 1280.50

✅ Nota 1003 processada
Fornecedor: Hardware & Redes SA
Valor total: R$ 8900.00


📊 PLANILHA GERADA COM SUCESSO

Arquivo: isabella_lima_aula11.xlsx
Notas processadas: 3


📋 RELATÓRIO DE NOTAS FISCAIS

🧾 Nota fiscal: 1001
Data de emissão: 10/10/2026
Fornecedor: Tech Solutions LTDA
CNPJ: 12.345.678/0001-90
Valor total: R$ 4500.00

🧾 Nota fiscal: 1002
Data de emissão: 12/10/2026
Fornecedor: Cloud Cloud Servicos
CNPJ: 98.765.432/0001-11
Valor total: R$ 1280.50

🧾 Nota fiscal: 1003
Data de emissão: 15/10/2026
Fornecedor: Hardware & Redes SA
CNPJ: 45.123.890/0001-55
Valor total: R$ 8900.00


✅ CONFERÊNCIA CONCLUÍDA
Notas processadas: 3
Valor total das compras: R$ 14680.50
Arquivo salvo: isabella_lima_aula11.xlsx
